# Demo SOKE ASL text to mesh on Google Colab

Notebook nay clone repo vao `/content/SOKE`, mount Google Drive, copy `last.ckpt` tu Drive ve `/content/last.ckpt`, copy cac asset can thiet ve local `/content`, sinh motion cho `TEXT = "I LOVE YOU"`, va render ra mot video mesh MP4 don gian.

Demo nay khong can `How2Sign.zip`, `t2m.tar.gz`, hay token cache. Cac file can co tren Drive: `archives/stats.zip`, `archives/mbart-h2s-csl-phoenix.zip`, `archives/smpl_models.zip`, `archives/tokenizer.ckpt`, va `experiments/mgpt/SOKE_COLAB_ASL_5K/checkpoints/last.ckpt`.


In [ ]:
# Sua cac bien nay truoc khi chay neu repo/path Drive cua ban khac.
GITHUB_REPO_URL = "https://github.com/KhoaLe1507/SOKE-Speech-to-SignLanguage-Realtim.git"
GITHUB_BRANCH = "main"

DRIVE_ROOT = "/content/drive/MyDrive/SOKE_COLAB"
LOCAL_ROOT = "/content/SOKE_COLAB_DATA"
LOCAL_ARCHIVE_ROOT = "/content/SOKE_COLAB_ARCHIVES"
DEMO_OUTPUT_ROOT = "/content/soke_demo_outputs"
REPO_DIR = "/content/SOKE"

CONFIG = "configs/soke_colab_asl_5k.yaml"
ASSETS_CONFIG = "configs/assets_colab.yaml"
TEXT = "I LOVE YOU"

CHECKPOINT_DRIVE = f"{DRIVE_ROOT}/experiments/mgpt/SOKE_COLAB_ASL_5K/checkpoints/last.ckpt"
CHECKPOINT_LOCAL = "/content/last.ckpt"


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!nvidia-smi


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--branch", GITHUB_BRANCH,
    GITHUB_REPO_URL, REPO_DIR,
], check=True)

os.chdir(REPO_DIR)
print("Repo dir:", os.getcwd())


In [ ]:
import sys
import subprocess

# Khong cai requirements.txt goc vi co Blender/bpy. Demo mesh don gian chi can cac dependency ben duoi.
commands = [
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    [sys.executable, "-m", "pip", "install", "-q", "imageio", "imageio-ffmpeg", "opencv-python-headless"],
]
for cmd in commands:
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("Dependency install done")


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)
drive_root = Path(DRIVE_ROOT)
drive_archive_root = drive_root / "archives"
local_root = Path(LOCAL_ROOT)
local_archive_root = Path(LOCAL_ARCHIVE_ROOT)
data_root = local_root / "data"
deps_root = local_root / "deps"
pretrained_root = local_root / "pretrained"
demo_output_root = Path(DEMO_OUTPUT_ROOT)

for path in [local_root, local_archive_root, data_root, deps_root, pretrained_root, demo_output_root]:
    path.mkdir(parents=True, exist_ok=True)

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

def copy_file_if_needed(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(f"Missing source file: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    should_copy = (
        (not dst.exists())
        or dst.stat().st_size != src.stat().st_size
        or src.stat().st_mtime > dst.stat().st_mtime
    )
    if should_copy:
        print("Copy:", src, "->", dst)
        shutil.copy2(src, dst)
    else:
        print("Local copy OK:", dst)
    return dst

def copy_archive_to_local(archive_names):
    drive_archive = first_existing(*[drive_archive_root / name for name in archive_names])
    if drive_archive is None:
        raise FileNotFoundError("Missing archive in Drive archives folder. Tried: " + ", ".join(archive_names))
    return copy_file_if_needed(drive_archive, local_archive_root / drive_archive.name)

def unpack_if_needed(expected_path, parent_dir, archive_names):
    expected_path = Path(expected_path)
    parent_dir = Path(parent_dir)
    if expected_path.exists():
        print("Found:", expected_path)
        return
    archive = copy_archive_to_local(archive_names)
    print("Extracting", archive, "->", parent_dir)
    parent_dir.mkdir(parents=True, exist_ok=True)
    if archive.suffix == ".zip":
        shutil.unpack_archive(str(archive), str(parent_dir))
    elif archive.name.endswith((".tar.gz", ".tgz")):
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(parent_dir)], check=True)
    else:
        raise ValueError(f"Unsupported archive format: {archive}")

# Asset toi thieu cho text -> mesh demo.
unpack_if_needed(data_root / "stats", data_root, ["stats.zip", "stats.tar.gz", "stats.tgz"])
unpack_if_needed(deps_root / "mbart-h2s-csl-phoenix", deps_root, ["mbart-h2s-csl-phoenix.zip", "mbart-h2s-csl-phoenix.tar.gz", "mbart-h2s-csl-phoenix.tgz"])
unpack_if_needed(deps_root / "smpl_models", deps_root, ["smpl_models.zip", "smpl_models.tar.gz", "smpl_models.tgz"])

# tokenizer.ckpt duoc luu nhu file rieng trong Drive archives.
tokenizer_src = copy_archive_to_local(["tokenizer.ckpt"])
tokenizer_local = copy_file_if_needed(tokenizer_src, pretrained_root / "tokenizer.ckpt")

# Checkpoint da train: copy tu Drive ve /content de inference nhanh hon doc truc tiep tu Drive.
checkpoint_local = copy_file_if_needed(CHECKPOINT_DRIVE, CHECKPOINT_LOCAL)

required_paths = [
    data_root / "stats" / "mean.pt",
    data_root / "stats" / "std.pt",
    pretrained_root / "tokenizer.ckpt",
    Path(CHECKPOINT_LOCAL),
    deps_root / "mbart-h2s-csl-phoenix" / "config.json",
    deps_root / "mbart-h2s-csl-phoenix" / "map_ids.pkl",
    deps_root / "mbart-h2s-csl-phoenix" / "pytorch_model.bin",
    deps_root / "mbart-h2s-csl-phoenix" / "sentencepiece.bpe.model",
    deps_root / "mbart-h2s-csl-phoenix" / "tokenizer.json",
    deps_root / "smpl_models" / "smplx" / "SMPLX_NEUTRAL.npz",
    deps_root / "smpl_models" / "smplx" / "SMPLX_to_J14.pkl",
    deps_root / "smpl_models" / "smplx" / "MANO_SMPLX_vertex_ids.pkl",
    deps_root / "smpl_models" / "smplx" / "SMPL-X__FLAME_vertex_ids.npy",
    deps_root / "smpl_models" / "smplx_vert_segmentation.json",
]
missing = [str(path) for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required demo files:\n" + "\n".join(missing))

Path("deps").mkdir(exist_ok=True)

def force_symlink(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.is_symlink() or dst.exists():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    os.symlink(src, dst, target_is_directory=True)

force_symlink(deps_root / "mbart-h2s-csl-phoenix", "deps/mbart-h2s-csl-phoenix")
force_symlink(deps_root / "smpl_models", "deps/smpl_models")

print("Demo assets OK")
print("Checkpoint local:", CHECKPOINT_LOCAL)
print("Tokenizer local:", tokenizer_local)


In [ ]:
import os
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from omegaconf import OmegaConf

os.chdir(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Demo chi can inference, khong can metric evaluator t2m. Patch truoc khi build model de tranh phu thuoc t2m.tar.gz.
from mGPT.models import base as base_module

def configure_metrics_noop(self):
    self.metrics = torch.nn.ModuleDict()

base_module.BaseModel.configure_metrics = configure_metrics_noop

from mGPT.config import get_module_config
from mGPT.models.build_model import build_model
from mGPT.utils.load_checkpoint import load_pretrained, load_pretrained_vae
from mGPT.utils.human_models import get_coord, smpl_x

try:
    OmegaConf.register_new_resolver("eval", eval)
except ValueError:
    pass

cfg_assets = OmegaConf.load(ASSETS_CONFIG)
cfg_base = OmegaConf.load(Path(cfg_assets.CONFIG_FOLDER) / "default.yaml")
cfg_exp = OmegaConf.merge(cfg_base, OmegaConf.load(CONFIG))
if not cfg_exp.FULL_CONFIG:
    cfg_exp = get_module_config(cfg_exp, cfg_assets.CONFIG_FOLDER)
cfg = OmegaConf.merge(cfg_exp, cfg_assets)

cfg.DEBUG = False
cfg.DEVICE = [0]
cfg.PRECISION = None
cfg.FOLDER = DEMO_OUTPUT_ROOT
cfg.TIME = "demo_text_to_mesh"
cfg.DATASET.NFEATS = 133
cfg.DATASET.TASK_PATH = ""
cfg.DATASET.H2S.MEAN_PATH = f"{LOCAL_ROOT}/data/stats/mean.pt"
cfg.DATASET.H2S.STD_PATH = f"{LOCAL_ROOT}/data/stats/std.pt"
cfg.TRAIN.STAGE = "lm_pretrain"
cfg.TRAIN.PRETRAINED = ""
cfg.TRAIN.PRETRAINED_VAE = f"{LOCAL_ROOT}/pretrained/tokenizer.ckpt"
cfg.TEST.CHECKPOINTS = CHECKPOINT_LOCAL
cfg.METRIC.TYPE = []
cfg.model.params.metrics_dict = []

def torch_load_trusted(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def load_h2s_stats(mean_path, std_path):
    mean = torch_load_trusted(mean_path, map_location="cpu")
    std = torch_load_trusted(std_path, map_location="cpu")
    mean = mean[(3 + 3 * 11):]
    mean = torch.cat([mean[:-20], mean[-10:]], dim=0)
    std = std[(3 + 3 * 11):]
    std = torch.cat([std[:-20], std[-10:]], dim=0)
    return mean.float(), std.float()

class DemoH2SDataModule:
    def __init__(self, cfg):
        self.cfg = cfg
        self.name = "demo_h2s"
        self.njoints = 22
        self.nfeats = 133
        mean, std = load_h2s_stats(cfg.DATASET.H2S.MEAN_PATH, cfg.DATASET.H2S.STD_PATH)
        self.hparams = SimpleNamespace(mean=mean, std=std, mean_eval=mean, std_eval=std)

    def feats2joints(self, features):
        mean = self.hparams.mean.to(features)
        std = self.hparams.std.to(features)
        features = features.float() * std + mean
        zero_pose = torch.zeros(*features.shape[:-1], 36, device=features.device, dtype=features.dtype)
        shape_param = torch.tensor([[[-0.07284723, 0.1795129, -0.27608207, 0.135155, 0.10748172,
                                      0.16037364, -0.01616933, -0.03450319, 0.01369138, 0.01108842]]],
                                   device=features.device, dtype=features.dtype)
        batch_size, frame_count = features.shape[:2]
        shape_param = shape_param.repeat(batch_size, frame_count, 1).view(batch_size * frame_count, -1)
        features = torch.cat([zero_pose, features], dim=-1).view(batch_size * frame_count, -1)
        vertices, joints = get_coord(
            root_pose=features[..., 0:3],
            body_pose=features[..., 3:66],
            lhand_pose=features[..., 66:111],
            rhand_pose=features[..., 111:156],
            jaw_pose=features[..., 156:159],
            shape=shape_param,
            expr=features[..., 159:169],
        )
        return vertices, joints

    def renorm4t2m(self, features):
        return features

datamodule = DemoH2SDataModule(cfg)
model = build_model(cfg, datamodule)
load_pretrained_vae(cfg, model)
load_pretrained(cfg, model, phase="test")
model = model.to(device).eval().float()

print("Model loaded")
print("Checkpoint:", cfg.TEST.CHECKPOINTS)
print("SMPL-X faces:", smpl_x.face.shape)


In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def decode_text_to_vertices(model, text):
    model.eval()
    src = ["how2sign"]
    name = ["demo_text"]
    gen_results = model.lm.generate_conditional(
        texts=[text],
        lengths=[0],
        stage="test",
        tasks=None,
        src=src,
        name=name,
    )

    body_tokens = gen_results["outputs_tokens"][0]
    lhand_tokens = gen_results["outputs_tokens_hand"][0] if gen_results["outputs_tokens_hand"] is not None else None
    rhand_tokens = gen_results["outputs_tokens_rhand"][0] if gen_results["outputs_tokens_rhand"] is not None else None

    token_lengths = [len(body_tokens)]
    if lhand_tokens is not None:
        token_lengths.append(len(lhand_tokens))
    if rhand_tokens is not None:
        token_lengths.append(len(rhand_tokens))
    max_len = max(max(token_lengths), 1) * 4

    feats = torch.zeros(1, max_len, 133, device=model.device, dtype=torch.float32)

    def decode_part(tokens, vae, nfeats):
        if tokens is None or len(tokens) <= 1:
            return torch.zeros(1, max_len, nfeats, device=model.device, dtype=torch.float32), 1
        tokens = torch.clamp(tokens.to(model.device).long(), 0, vae.code_num - 1)
        motion = vae.decode(tokens).float()
        part_len = motion.shape[1]
        if part_len < max_len:
            motion = F.pad(motion, (0, 0, 0, max_len - part_len), mode="replicate")
        elif part_len > max_len:
            motion = motion[:, :max_len]
            part_len = max_len
        return motion, part_len

    body_motion, body_len = decode_part(body_tokens, model.vae, model.vae.nfeats)
    feats[:, :, :30] = body_motion[:, :, :30]
    feats[:, :, -13:] = body_motion[:, :, 30:43]

    output_len = body_len
    if hasattr(model, "hand_vae") and lhand_tokens is not None:
        lhand_motion, lhand_len = decode_part(lhand_tokens, model.hand_vae, model.hand_vae.nfeats)
        feats[:, :, 30:30 + model.hand_vae.nfeats] = lhand_motion
        output_len = max(output_len, lhand_len)

    if hasattr(model, "rhand_vae") and rhand_tokens is not None:
        rhand_motion, rhand_len = decode_part(rhand_tokens, model.rhand_vae, model.rhand_vae.nfeats)
        feats[:, :, 75:75 + model.rhand_vae.nfeats] = rhand_motion
        output_len = max(output_len, rhand_len)

    vertices, joints = datamodule.feats2joints(feats[:, :output_len])
    vertices = vertices.view(1, output_len, -1, 3)[0].detach().cpu().numpy()
    joints = joints.view(1, output_len, -1, 3)[0].detach().cpu().numpy()
    feats = feats[:, :output_len].detach().cpu().numpy()

    print("Text:", text)
    print("Token counts:", {
        "body": len(body_tokens),
        "lhand": len(lhand_tokens) if lhand_tokens is not None else 0,
        "rhand": len(rhand_tokens) if rhand_tokens is not None else 0,
    })
    print("Generated frames:", output_len)
    return vertices, joints, feats, gen_results

vertices, joints, feats, gen_results = decode_text_to_vertices(model, TEXT)


In [ ]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"

from base64 import b64encode
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
import pyrender
import shutil
import trimesh
from IPython.display import HTML, display
from tqdm.auto import tqdm

render_root = Path(DEMO_OUTPUT_ROOT)
render_root.mkdir(parents=True, exist_ok=True)
safe_text = "".join(ch if ch.isalnum() else "_" for ch in TEXT).strip("_")[:48] or "text"
output_mp4 = render_root / f"soke_mesh_{safe_text}.mp4"
first_frame_png = render_root / f"soke_mesh_{safe_text}_first_frame.png"
vertices_npy = render_root / f"soke_mesh_{safe_text}_vertices.npy"
np.save(vertices_npy, vertices)

fps = 18
height, width = 512, 512
max_render_frames = 160
vertices_to_render = vertices[:max_render_frames]
faces = smpl_x.face
cam_trans = np.array([-2.6177440e-03, 0.1, -13.0], dtype=np.float32)

renderer = pyrender.OffscreenRenderer(viewport_width=width, viewport_height=height, point_size=1.0)

def render_mesh_frame(frame_vertices):
    background = np.ones((height, width, 3), dtype=np.uint8) * 255
    mesh = trimesh.Trimesh(frame_vertices.copy(), faces, process=False)
    rot = trimesh.transformations.rotation_matrix(np.radians(180), [1, 0, 0])
    mesh.apply_transform(rot)

    material = pyrender.MetallicRoughnessMaterial(
        metallicFactor=0.0,
        alphaMode="OPAQUE",
        baseColorFactor=(0.78, 0.9, 0.96, 1.0),
    )
    scene = pyrender.Scene(ambient_light=(0.35, 0.35, 0.35), bg_color=(1.0, 1.0, 1.0, 1.0))
    scene.add(pyrender.Mesh.from_trimesh(mesh, material=material, smooth=False), "mesh")

    camera_center = [width / 2.0, height / 2.0]
    camera_pose = np.eye(4)
    camera_pose[:3, 3] = cam_trans
    camera_pose[:3, :3] = [[1, 0, 0], [0, -1, 0], [0, 0, -1]]
    camera = pyrender.camera.IntrinsicsCamera(fx=5000, fy=5000, cx=camera_center[0], cy=camera_center[1])
    scene.add(camera, pose=camera_pose)

    light = pyrender.DirectionalLight(color=[1, 1, 1], intensity=500)
    scene.add(light, pose=np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]]))

    rgb, depth = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
    rgb = rgb[:, :, :3].astype(np.float32)
    valid_mask = (depth > 0)[:, :, None]
    frame = rgb * valid_mask + background * (1 - valid_mask)
    return frame.astype(np.uint8)

try:
    writer = imageio.get_writer(str(output_mp4), fps=fps, codec="libx264", quality=8, macro_block_size=1)
    for idx, frame_vertices in enumerate(tqdm(vertices_to_render, desc="Rendering mesh")):
        frame = render_mesh_frame(frame_vertices)
        if idx == 0:
            imageio.imwrite(first_frame_png, frame)
        writer.append_data(frame)
    writer.close()
finally:
    renderer.delete()

print("Saved MP4:", output_mp4)
print("Saved first frame:", first_frame_png)
print("Saved vertices:", vertices_npy)

# Copy ket qua ve Drive de khong mat khi Colab runtime bi reset.
drive_demo_out = Path(DRIVE_ROOT) / "results" / "demo_text_to_mesh"
drive_demo_out.mkdir(parents=True, exist_ok=True)
for path in [output_mp4, first_frame_png, vertices_npy]:
    shutil.copy2(path, drive_demo_out / path.name)
print("Copied demo outputs to:", drive_demo_out)

video_bytes = output_mp4.read_bytes()
display(HTML(f'''<video width="512" controls><source src="data:video/mp4;base64,{b64encode(video_bytes).decode()}" type="video/mp4"></video>'''))


In [ ]:
from pathlib import Path

print("Local outputs:")
for path in sorted(Path(DEMO_OUTPUT_ROOT).glob("*")):
    print(path, f"{path.stat().st_size / (1024 ** 2):.2f} MiB")

print("Drive outputs:")
for path in sorted((Path(DRIVE_ROOT) / "results" / "demo_text_to_mesh").glob("*")):
    print(path, f"{path.stat().st_size / (1024 ** 2):.2f} MiB")
